# JAO Feature Engineering

JAO publishes the day-ahead forecast at 10:30 on D-1, so every series is already known for delivery day D. Features stay on hour t. A 24-hour change is the value at t minus the value 24 hours earlier. `create_all_jao_features` builds every series once. Each model family is saved in two scopes: Poland, and Poland plus the foreign addition. That is four files. `select_features(df, model, scope)` keeps one of them: `model` is `"trees"` or `"nn"`, and `scope` is `"pl"` or `"pl_foreign"`. The second-ring series for a later PCA are appended to both foreign files under a `pca_` prefix. No components are computed here.

**Inputs**

* One merged JAO frame: core net positions, the Polish min/max net-position limits, reference-program border flows, D2CF generation and vertical load, and the Polish allocation constraints.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.file_io import (
    load_merged_data,
    save_merged_data,
    inspect_merged_file,
)
from src.utils.feature_engineering.fe_jao import (
    COLUMNS,
    PCA_GROUPS,
    create_all_jao_features,
    select_dataset,
    select_features,
    select_pca_group,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

data_jao = load_merged_data(
    DATA_DIR / "preprocessed" / "jao" / date_range / "jao.parquet"
)

## 1. Input Columns

The merged JAO frame, before any feature is derived from it.

In [3]:
print(list(data_jao.columns))

['allocation_constraint_PL_import', 'allocation_constraint_PL_export', 'd2cf_verticalLoad_AT', 'd2cf_verticalLoad_BE', 'd2cf_verticalLoad_CZ', 'd2cf_verticalLoad_DE', 'd2cf_verticalLoad_FR', 'd2cf_verticalLoad_HR', 'd2cf_verticalLoad_HU', 'd2cf_verticalLoad_NL', 'd2cf_verticalLoad_PL', 'd2cf_verticalLoad_RO', 'd2cf_verticalLoad_SI', 'd2cf_verticalLoad_SK', 'd2cf_generation_AT', 'd2cf_generation_BE', 'd2cf_generation_CZ', 'd2cf_generation_DE', 'd2cf_generation_FR', 'd2cf_generation_HR', 'd2cf_generation_HU', 'd2cf_generation_NL', 'd2cf_generation_PL', 'd2cf_generation_RO', 'd2cf_generation_SI', 'd2cf_generation_SK', 'd2cf_coreNetPosition_AT', 'd2cf_coreNetPosition_BE', 'd2cf_coreNetPosition_CZ', 'd2cf_coreNetPosition_DE', 'd2cf_coreNetPosition_FR', 'd2cf_coreNetPosition_HR', 'd2cf_coreNetPosition_HU', 'd2cf_coreNetPosition_NL', 'd2cf_coreNetPosition_PL', 'd2cf_coreNetPosition_RO', 'd2cf_coreNetPosition_SI', 'd2cf_coreNetPosition_SK', 'd2cf_verticalLoad_50Hertz', 'd2cf_verticalLoad_Ampri

## 2. Features

**Poland and Polish borders.** 14 columns for trees, 10 for neural networks. Core net position `np_PL`, day-ahead generation `gen_PL`, the import and export limits `min_PL` and `max_PL`, and the allocation constraints `ac_PL_import` and `ac_PL_export`. Border flows are signed so that positive means flow into Poland: `refprog_border_DE_PL` as published, and the reverse of `refprog_border_PL_CZ`, `refprog_border_PL_SK` and `refprog_border_PL_UA`, stored as `flow_DE_PL`, `flow_CZ_PL`, `flow_SK_PL` and `flow_UA_PL`.

Trees also keep `headroom_import_PL` (`np_PL − min_PL`), `headroom_export_PL` (`max_PL − np_PL`), and the 24-hour changes `np_PL_delta24` and `gen_PL_delta24`.

**Foreign addition.** 11 further columns for trees, 9 for neural networks. Germany (`np_DE`, `gen_DE`), 50Hertz (`np_50Hz`, `gen_50Hz`, `vload_50Hz`), Czechia (`np_CZ`, `gen_CZ`) and Slovakia (`np_SK`, `gen_SK`). Trees also keep `gen_DE_delta24` and `gen_50Hz_delta24`.

**PCA inputs.** Second-ring zones AT, BE, FR, HR, HU, NL, RO and SI. The other German bidding-zone parts (Amprion, TenneT, TransnetBW, Creos) stay out of this ring, because they belong to Germany, and 50Hertz is already a feature above. `PCA_GROUPS` lists the blocks, and `select_pca_group(df, name)` returns the columns of one block. The later PCA keeps two components of net position and one each of generation and vertical load.

* `pca_np`: core net position of the eight zones. Later `PC_np_1`, `PC_np_2`.
* `pca_gen`: day-ahead generation of the same zones. Later `PC_gen_1`.
* `pca_load`: vertical load of the same zones. Later `PC_load_1`.

In [4]:
features = create_all_jao_features(data_jao)

In [5]:
print(features.shape)
print(list(features.columns))

for group in PCA_GROUPS:
    cols = list(select_pca_group(features, group).columns)
    print(group, len(cols))
    print(cols)

(33263, 49)
['np_PL', 'gen_PL', 'min_PL', 'max_PL', 'ac_PL_import', 'ac_PL_export', 'headroom_import_PL', 'headroom_export_PL', 'np_PL_delta24', 'gen_PL_delta24', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL', 'np_DE', 'gen_DE', 'gen_DE_delta24', 'np_50Hz', 'gen_50Hz', 'vload_50Hz', 'gen_50Hz_delta24', 'np_CZ', 'gen_CZ', 'np_SK', 'gen_SK', 'pca_np_AT', 'pca_np_BE', 'pca_np_FR', 'pca_np_HR', 'pca_np_HU', 'pca_np_NL', 'pca_np_RO', 'pca_np_SI', 'pca_gen_AT', 'pca_gen_BE', 'pca_gen_FR', 'pca_gen_HR', 'pca_gen_HU', 'pca_gen_NL', 'pca_gen_RO', 'pca_gen_SI', 'pca_load_AT', 'pca_load_BE', 'pca_load_FR', 'pca_load_HR', 'pca_load_HU', 'pca_load_NL', 'pca_load_RO', 'pca_load_SI']
pca_np 8
['pca_np_AT', 'pca_np_BE', 'pca_np_FR', 'pca_np_HR', 'pca_np_HU', 'pca_np_NL', 'pca_np_RO', 'pca_np_SI']
pca_gen 8
['pca_gen_AT', 'pca_gen_BE', 'pca_gen_FR', 'pca_gen_HR', 'pca_gen_HU', 'pca_gen_NL', 'pca_gen_RO', 'pca_gen_SI']
pca_load 8
['pca_load_AT', 'pca_load_BE', 'pca_load_FR', 'pca_load_HR', 'pca

## 3. The Four Subsets

`COLUMNS` lists the four combinations. `pl` is Poland. `pl_foreign` is Poland plus the foreign addition.

Trees keep 14 Polish columns, and 25 with the foreign addition.

Neural networks keep 10 Polish columns. Headroom and the 24-hour changes stay in the tree set. With the foreign addition that is 19 columns. `gen_DE_delta24` and `gen_50Hz_delta24` stay in the tree set.

`select_dataset` is what gets saved. For `"pl"` it is the same frame. For `"pl_foreign"` it appends the three PCA blocks, the same columns for both families.

In [6]:
for model, scopes in COLUMNS.items():
    for scope in scopes:
        subset = select_features(features, model, scope)
        saved = select_dataset(features, model, scope)
        print(f"=== {model}/{scope} ===")
        print(subset.shape[1], "features,", saved.shape[1], "saved")
        print(list(subset.columns))

=== trees/pl ===
14 features, 14 saved
['np_PL', 'gen_PL', 'min_PL', 'max_PL', 'ac_PL_import', 'ac_PL_export', 'headroom_import_PL', 'headroom_export_PL', 'np_PL_delta24', 'gen_PL_delta24', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL']
=== trees/pl_foreign ===
25 features, 49 saved
['np_PL', 'gen_PL', 'min_PL', 'max_PL', 'ac_PL_import', 'ac_PL_export', 'headroom_import_PL', 'headroom_export_PL', 'np_PL_delta24', 'gen_PL_delta24', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL', 'np_DE', 'gen_DE', 'gen_DE_delta24', 'np_50Hz', 'gen_50Hz', 'vload_50Hz', 'gen_50Hz_delta24', 'np_CZ', 'gen_CZ', 'np_SK', 'gen_SK']
=== nn/pl ===
10 features, 10 saved
['np_PL', 'gen_PL', 'min_PL', 'max_PL', 'ac_PL_import', 'ac_PL_export', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL']
=== nn/pl_foreign ===
19 features, 43 saved
['np_PL', 'gen_PL', 'min_PL', 'max_PL', 'ac_PL_import', 'ac_PL_export', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL', 'np_DE', 'gen_DE', 'np_50Hz', 'g

## 4. Conclusions

Poland and Poland plus the foreign addition are each saved for trees and for neural networks, four files in total. The foreign files also hold the `pca_` inputs. `select_pca_group` takes one name from `PCA_GROUPS` and returns that block. Files go to `data/features/jao/{trees|nn}/{pl|pl_foreign}/`, and each file is verified by reading it back.

## 5. Saving Feature Data

In [7]:
for model, scopes in COLUMNS.items():
    for scope in scopes:
        path = DATA_DIR / "features" / "jao" / model / scope / f"{date_range}.parquet"
        save_merged_data(select_dataset(features, model, scope), path)
        inspect_merged_file(path)

  [FILE] data/features/jao/trees/pl/2022-12-01_2026-09-16.parquet | shape: (33263, 14) | idx: 2022-11-30 23:00:00+00:00-2026-09-16 21:00:00+00:00 | cols: 14 | NaN: 336
  [FILE] data/features/jao/trees/pl_foreign/2022-12-01_2026-09-16.parquet | shape: (33263, 49) | idx: 2022-11-30 23:00:00+00:00-2026-09-16 21:00:00+00:00 | cols: 49 | NaN: 1224
  [FILE] data/features/jao/nn/pl/2022-12-01_2026-09-16.parquet | shape: (33263, 10) | idx: 2022-11-30 23:00:00+00:00-2026-09-16 21:00:00+00:00 | cols: 10 | NaN: 192
  [FILE] data/features/jao/nn/pl_foreign/2022-12-01_2026-09-16.parquet | shape: (33263, 43) | idx: 2022-11-30 23:00:00+00:00-2026-09-16 21:00:00+00:00 | cols: 43 | NaN: 984
